# Georeferenced image-level coverage

Requires per_image_gd_coverage.csv. These are sampled image locations, not independently validated continuous field disease maps.

Source: `graindiscoloration (2).ipynb`, zero-based cell 54.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
"""Create publication-ready georeferenced GD coverage maps for the 60-image test set.

Input
-----
The script uses ``per_image_gd_coverage.csv`` from
``gd_image_level_coverage_seed42.zip``. Latitude, longitude, and altitude are
parsed directly from each test-image name, so no train/validation prediction
table is required.

Outputs
-------
1. test60_georeferenced_coverage_overall.{png,pdf}
2. test60_georeferenced_coverage_by_area.{png,pdf}
3. test60_georeferenced_coverage.csv
4. test60_georeferenced_mapping_summary.json
5. gd_test60_georeferenced_mapping.zip
"""

from __future__ import annotations

import io
import json
import re
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize
from matplotlib.ticker import FormatStrFormatter


# Set this to a CSV or ZIP path only if automatic discovery selects the wrong file.
# Kaggle example:
# COVERAGE_SOURCE = "/kaggle/input/your-dataset/gd_image_level_coverage_seed42.zip"
COVERAGE_SOURCE = None

EXPECTED_TEST_IMAGES = 60
VALIDATION_SELECTED_THRESHOLD = 0.60

if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/gd_test60_georeferenced_mapping")
else:
    OUTPUT_DIR = Path.cwd() / "gd_test60_georeferenced_mapping"


def read_coverage_source(path: Path) -> tuple[pd.DataFrame, str]:
    """Read per-image coverage data from a CSV, ZIP, or directory."""
    path = path.expanduser().resolve()

    if path.is_dir():
        matches = sorted(path.rglob("per_image_gd_coverage.csv"))
        if not matches:
            raise FileNotFoundError(f"No per_image_gd_coverage.csv inside {path}")
        path = matches[0]

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path), str(path)

    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path) as archive:
            members = [
                name
                for name in archive.namelist()
                if Path(name).name == "per_image_gd_coverage.csv"
            ]
            if len(members) != 1:
                raise RuntimeError(
                    f"Expected one per_image_gd_coverage.csv in {path}, found {len(members)}"
                )
            with archive.open(members[0]) as raw:
                frame = pd.read_csv(io.TextIOWrapper(raw, encoding="utf-8"))
        return frame, f"{path}::{members[0]}"

    raise ValueError(f"Unsupported source: {path}")


def discover_coverage_source() -> Path:
    """Find the existing 60-image coverage CSV/ZIP in Kaggle or locally."""
    if COVERAGE_SOURCE is not None:
        source = Path(COVERAGE_SOURCE)
        if not source.exists():
            raise FileNotFoundError(f"COVERAGE_SOURCE does not exist: {source}")
        return source

    roots = [
        Path("/kaggle/working"),
        Path("/kaggle/input"),
        Path.cwd(),
        Path.cwd() / "upload",
    ]
    roots = [root for root in roots if root.exists()]

    # Prefer the raw CSV; otherwise use the exact result ZIP.
    csv_candidates: list[Path] = []
    zip_candidates: list[Path] = []
    for root in roots:
        csv_candidates.extend(root.rglob("per_image_gd_coverage.csv"))
        zip_candidates.extend(root.rglob("gd_image_level_coverage_seed42.zip"))

    def unique(paths: list[Path]) -> list[Path]:
        seen = set()
        result = []
        for item in paths:
            resolved = item.resolve()
            if resolved not in seen:
                seen.add(resolved)
                result.append(resolved)
        return result

    csv_candidates = unique(csv_candidates)
    zip_candidates = unique(zip_candidates)

    if csv_candidates:
        return csv_candidates[0]
    if zip_candidates:
        return zip_candidates[0]

    raise FileNotFoundError(
        "Could not find per_image_gd_coverage.csv or "
        "gd_image_level_coverage_seed42.zip. Attach the result ZIP to the "
        "Kaggle notebook or set COVERAGE_SOURCE manually."
    )


GPS_PATTERN = re.compile(
    r"lat_(?P<latitude>-?\d+(?:\.\d+)?)_"
    r"lon_(?P<longitude>-?\d+(?:\.\d+)?)_"
    r"alt_(?P<altitude>-?\d+(?:\.\d+)?)m",
    flags=re.IGNORECASE,
)


def extract_gps(image_name: str) -> pd.Series:
    """Extract GPS values embedded in a DJI image name."""
    match = GPS_PATTERN.search(str(image_name))
    if match is None:
        return pd.Series(
            {"latitude": np.nan, "longitude": np.nan, "altitude_m": np.nan}
        )
    values = {key: float(value) for key, value in match.groupdict().items()}
    return pd.Series(
        {
            "latitude": values["latitude"],
            "longitude": values["longitude"],
            "altitude_m": values["altitude"],
        }
    )


def prepare_test_data(frame: pd.DataFrame) -> tuple[pd.DataFrame, float]:
    required = {
        "image",
        "ground_truth_coverage_percent",
        "predicted_coverage_percent",
    }
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    data = frame.copy()
    if len(data) != EXPECTED_TEST_IMAGES:
        raise RuntimeError(
            f"Expected exactly {EXPECTED_TEST_IMAGES} test images, found {len(data)}. "
            "Stop here to avoid accidentally mapping train/validation images."
        )
    if data["image"].duplicated().any():
        duplicates = data.loc[data["image"].duplicated(), "image"].tolist()
        raise RuntimeError(f"Duplicate test image names found: {duplicates[:5]}")

    gps = data["image"].apply(extract_gps)
    data = pd.concat([data, gps], axis=1)
    if data[["latitude", "longitude"]].isna().any().any():
        failed = data.loc[
            data[["latitude", "longitude"]].isna().any(axis=1), "image"
        ].tolist()
        raise RuntimeError(f"Could not parse GPS from {len(failed)} image names: {failed[:5]}")

    for column in ["ground_truth_coverage_percent", "predicted_coverage_percent"]:
        data[column] = pd.to_numeric(data[column], errors="raise")

    data["signed_error_percentage_points"] = (
        data["predicted_coverage_percent"] - data["ground_truth_coverage_percent"]
    )
    data["absolute_error_percentage_points"] = data[
        "signed_error_percentage_points"
    ].abs()

    # The two UAV acquisition areas are separated by the largest longitude gap.
    sorted_longitudes = np.sort(data["longitude"].unique())
    gaps = np.diff(sorted_longitudes)
    if len(gaps) == 0:
        raise RuntimeError("At least two distinct longitude values are required.")
    split_index = int(np.argmax(gaps))
    longitude_cutoff = float(
        (sorted_longitudes[split_index] + sorted_longitudes[split_index + 1]) / 2
    )
    data["acquisition_area"] = np.where(
        data["longitude"] < longitude_cutoff,
        "Acquisition area A",
        "Acquisition area B",
    )

    area_counts = data["acquisition_area"].value_counts()
    if len(area_counts) != 2 or area_counts.min() < 5:
        raise RuntimeError(
            "Automatic spatial separation did not produce two usable acquisition areas. "
            f"Counts: {area_counts.to_dict()}"
        )

    return data, longitude_cutoff


def configure_geo_axis(ax: plt.Axes, subset: pd.DataFrame) -> None:
    """Use readable coordinate labels and an approximately geographic aspect ratio."""
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.4f"))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.4f"))
    ax.tick_params(axis="x", rotation=25)
    mean_latitude = float(subset["latitude"].mean())
    ax.set_aspect(1.0 / np.cos(np.deg2rad(mean_latitude)), adjustable="box")
    ax.grid(True, linestyle=":", linewidth=0.7, alpha=0.45)


def add_extent_padding(ax: plt.Axes, subset: pd.DataFrame, fraction: float = 0.12) -> None:
    lon_min, lon_max = subset["longitude"].min(), subset["longitude"].max()
    lat_min, lat_max = subset["latitude"].min(), subset["latitude"].max()
    lon_pad = max((lon_max - lon_min) * fraction, 0.00002)
    lat_pad = max((lat_max - lat_min) * fraction, 0.00002)
    ax.set_xlim(lon_min - lon_pad, lon_max + lon_pad)
    ax.set_ylim(lat_min - lat_pad, lat_max + lat_pad)


def plot_overall_map(data: pd.DataFrame, output_dir: Path) -> None:
    coverage_min = float(
        min(
            data["ground_truth_coverage_percent"].min(),
            data["predicted_coverage_percent"].min(),
        )
    )
    coverage_max = float(
        max(
            data["ground_truth_coverage_percent"].max(),
            data["predicted_coverage_percent"].max(),
        )
    )
    error_max = float(data["absolute_error_percentage_points"].max())
    coverage_norm = Normalize(vmin=coverage_min, vmax=coverage_max)
    error_norm = Normalize(vmin=0.0, vmax=max(error_max, 1e-8))

    fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.6), layout="constrained")
    panels = [
        ("ground_truth_coverage_percent", "Ground-truth coverage", "viridis", coverage_norm),
        ("predicted_coverage_percent", "Predicted coverage", "viridis", coverage_norm),
        (
            "absolute_error_percentage_points",
            "Absolute coverage error",
            "magma",
            error_norm,
        ),
    ]

    artists = []
    for label, (column, title, cmap, norm) in zip(["(a)", "(b)", "(c)"], panels):
        ax = axes[len(artists)]
        artist = ax.scatter(
            data["longitude"],
            data["latitude"],
            c=data[column],
            cmap=cmap,
            norm=norm,
            s=58,
            edgecolors="white",
            linewidths=0.55,
            alpha=0.95,
        )
        artists.append(artist)
        ax.set_title(f"{label} {title}", fontweight="bold")
        configure_geo_axis(ax, data)
        add_extent_padding(ax, data, fraction=0.08)

    coverage_bar = fig.colorbar(
        artists[0], ax=axes[:2], location="bottom", shrink=0.72, pad=0.14
    )
    coverage_bar.set_label("Image-level GD coverage (%)")
    error_bar = fig.colorbar(
        artists[2], ax=axes[2], location="bottom", shrink=0.82, pad=0.14
    )
    error_bar.set_label("Absolute error (percentage points)")

    fig.suptitle(
        "Georeferenced Image-Level GD Coverage on the 60-Image Test Set",
        fontsize=15,
        fontweight="bold",
    )
    # The PDF remains vector-based; 300 dpi keeps the PNG high quality and portable.
    fig.savefig(output_dir / "test60_georeferenced_coverage_overall.png", dpi=300)
    fig.savefig(output_dir / "test60_georeferenced_coverage_overall.pdf")
    plt.show()
    plt.close(fig)


def plot_area_wise_map(data: pd.DataFrame, output_dir: Path) -> None:
    coverage_min = float(
        min(
            data["ground_truth_coverage_percent"].min(),
            data["predicted_coverage_percent"].min(),
        )
    )
    coverage_max = float(
        max(
            data["ground_truth_coverage_percent"].max(),
            data["predicted_coverage_percent"].max(),
        )
    )
    error_max = float(data["absolute_error_percentage_points"].max())
    coverage_norm = Normalize(vmin=coverage_min, vmax=coverage_max)
    error_norm = Normalize(vmin=0.0, vmax=max(error_max, 1e-8))

    areas = ["Acquisition area A", "Acquisition area B"]
    panels = [
        ("ground_truth_coverage_percent", "Ground-truth coverage", "viridis", coverage_norm),
        ("predicted_coverage_percent", "Predicted coverage", "viridis", coverage_norm),
        (
            "absolute_error_percentage_points",
            "Absolute coverage error",
            "magma",
            error_norm,
        ),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15.5, 9.2), layout="constrained")
    coverage_artist = None
    error_artist = None

    for row, area in enumerate(areas):
        subset = data.loc[data["acquisition_area"] == area].copy()
        for col, (column, title, cmap, norm) in enumerate(panels):
            ax = axes[row, col]
            artist = ax.scatter(
                subset["longitude"],
                subset["latitude"],
                c=subset[column],
                cmap=cmap,
                norm=norm,
                s=72,
                edgecolors="white",
                linewidths=0.6,
                alpha=0.95,
            )
            if col < 2:
                coverage_artist = artist
            else:
                error_artist = artist
            panel_label = chr(ord("a") + row * 3 + col)
            ax.set_title(
                f"({panel_label}) {area}: {title}\n(n = {len(subset)})",
                fontweight="bold",
            )
            configure_geo_axis(ax, subset)
            add_extent_padding(ax, subset)

    coverage_bar = fig.colorbar(
        coverage_artist,
        ax=axes[:, :2],
        location="bottom",
        shrink=0.70,
        pad=0.10,
    )
    coverage_bar.set_label("Image-level GD coverage (%)")
    error_bar = fig.colorbar(
        error_artist,
        ax=axes[:, 2],
        location="bottom",
        shrink=0.78,
        pad=0.10,
    )
    error_bar.set_label("Absolute error (percentage points)")

    fig.suptitle(
        "Georeferenced Image-Level GD Coverage by Acquisition Area",
        fontsize=15,
        fontweight="bold",
    )
    fig.savefig(output_dir / "test60_georeferenced_coverage_by_area.png", dpi=300)
    fig.savefig(output_dir / "test60_georeferenced_coverage_by_area.pdf")
    plt.show()
    plt.close(fig)


def main() -> None:
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10.5,
            "axes.titlepad": 9,
            "savefig.bbox": "tight",
            "savefig.facecolor": "white",
        }
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    source_path = discover_coverage_source()
    raw_frame, source_description = read_coverage_source(source_path)
    data, longitude_cutoff = prepare_test_data(raw_frame)
    data = data.sort_values(["acquisition_area", "longitude", "latitude"]).reset_index(drop=True)

    output_csv = OUTPUT_DIR / "test60_georeferenced_coverage.csv"
    data.to_csv(output_csv, index=False)

    plot_overall_map(data, OUTPUT_DIR)
    plot_area_wise_map(data, OUTPUT_DIR)

    summary = {
        "evaluation_subset": "test only",
        "number_of_test_images": int(len(data)),
        "evaluation_condition": "no TTA",
        "validation_selected_threshold": VALIDATION_SELECTED_THRESHOLD,
        "coverage_denominator": "full image area",
        "coordinate_source": "latitude/longitude parsed from test-image names",
        "source": source_description,
        "mae_percentage_points": float(data["absolute_error_percentage_points"].mean()),
        "rmse_percentage_points": float(
            np.sqrt(np.mean(np.square(data["signed_error_percentage_points"])))
        ),
        "mean_signed_error_percentage_points": float(
            data["signed_error_percentage_points"].mean()
        ),
        "mean_ground_truth_coverage_percent": float(
            data["ground_truth_coverage_percent"].mean()
        ),
        "mean_predicted_coverage_percent": float(
            data["predicted_coverage_percent"].mean()
        ),
        "longitude_cutoff_used_only_to_separate_acquisition_areas": longitude_cutoff,
        "images_per_acquisition_area": {
            key: int(value)
            for key, value in data["acquisition_area"].value_counts().sort_index().items()
        },
    }
    with open(
        OUTPUT_DIR / "test60_georeferenced_mapping_summary.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(summary, handle, indent=2)

    archive_path = OUTPUT_DIR / "gd_test60_georeferenced_mapping.zip"
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT_DIR.iterdir()):
            if path.is_file() and path != archive_path:
                archive.write(path, arcname=path.name)

    print("\n✅ Test-only georeferenced mapping completed")
    print(f"Source: {source_description}")
    print(f"Images used: {len(data)} (expected: {EXPECTED_TEST_IMAGES})")
    print(f"Area counts: {summary['images_per_acquisition_area']}")
    print(f"Output folder: {OUTPUT_DIR}")
    print(f"Download ZIP: {archive_path}")

    try:
        from IPython.display import FileLink, display

        display(FileLink(str(archive_path)))
    except Exception:
        pass


if __name__ == "__main__":
    main()